BIBLIOTECAS

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

# Preprocesamiento de datos
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Algoritmos de regresion
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor

# metricas de regresion
from sklearn.metrics import mean_squared_error, r2_score


PREPROCESADO DE DATOS

In [3]:
df = pd.read_csv('job_salary_prediction_dataset.csv')
display(df)

print('Dimensiones:', df.shape)
print('\nTipos de datos:')
print(df.dtypes)
print('\nValores nulos por columna:')
print(df.isna().sum())
print('\nFilas duplicadas:', df.duplicated().sum())
print('\nResumen de la variable objetivo :')
print(df['salary'].describe())

df.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069
...,...,...,...,...,...,...,...,...,...,...
249995,Software Engineer,17,PhD,2,Telecom,Enterprise,India,No,1,127791
249996,Frontend Developer,20,PhD,7,Telecom,Startup,Remote,No,2,154593
249997,Business Analyst,1,Bachelor,12,Retail,Enterprise,India,Yes,0,75988
249998,Data Scientist,0,High School,2,Consulting,Small,Sweden,Hybrid,5,90467


Dimensiones: (250000, 10)

Tipos de datos:
job_title           object
experience_years     int64
education_level     object
skills_count         int64
industry            object
company_size        object
location            object
remote_work         object
certifications       int64
salary               int64
dtype: object

Valores nulos por columna:
job_title           0
experience_years    0
education_level     0
skills_count        0
industry            0
company_size        0
location            0
remote_work         0
certifications      0
salary              0
dtype: int64

Filas duplicadas: 0

Resumen de la variable objetivo :
count    250000.000000
mean     145718.080524
std       37407.952729
min       31867.000000
25%      119358.000000
50%      143453.000000
75%      169492.000000
max      333046.000000
Name: salary, dtype: float64


,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [4]:
# Borrar columnas que no nos digan nada
# df = df.drop("columna_a_borrar",axis=1)


# Variables predictoras y variable objetivo
X = df.drop('salary', axis=1)
y = df['salary']


# Columnas numéricas y categóricas
cat_cols = X.select_dtypes(include=['object']).columns
num_cols = X.select_dtypes(exclude=['object']).columns

# median = mediana ; mean = media

# Pipelines para preprocesamiento
# primero los NaN les pones la mediana
# después los escalas(media 0) desviación típica 1
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),      
    ('scaler', StandardScaler())                        
])


# si a una fila le falta algo le pones 'most_frequent'
# OHE , min_frequency=20 (tiene que haber al menos 20 para que se cree una columna), 
#  sino lo agrupa en una nueva que se llama 'nombre_columna_infrequent_if_exist'
# ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20)) 
# si quieres que ignore algo:
# ('onehot', OneHotEncoder(handle_unknown='ignore')) 
# para que haga todo:
# ('onehot', OneHotEncoder(sparse_output=False))   
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20))   
])

# Combinamos los transformadores en un ColumnTransformer
# habla entre las cols y los transformadores y lo combina
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# Dividimos el dataset en entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Muestras para entrenar: {X_train.shape[0]}')
print(f'Muestras para validar: {X_test.shape[0]}')
print(f'Número de variables numéricas: {len(num_cols)}')
print(f'Número de variables categóricas: {len(cat_cols)}')

Muestras para entrenar: 200000
Muestras para validar: 50000
Número de variables numéricas: 3
Número de variables categóricas: 6


In [5]:
# Procesa las columnas(numericas y categóricas)
# fit_transformed = aprende las medias y medianas y aplica
# transform = solo aplica con lo que ya has aprendido
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print('Forma del conjunto de entrenamiento tras el preprocesado:', X_train_transformed.shape)
print('Forma del conjunto de prueba tras el preprocesado:', X_test_transformed.shape)
print('Número total de variables generadas:', len(preprocessor.get_feature_names_out()))

feature_names = preprocessor.get_feature_names_out()
print('\nPrimeras 15 variables transformadas:')
print(feature_names[:15])

Forma del conjunto de entrenamiento tras el preprocesado: (200000, 48)
Forma del conjunto de prueba tras el preprocesado: (50000, 48)
Número total de variables generadas: 48

Primeras 15 variables transformadas:
['num__experience_years' 'num__skills_count' 'num__certifications'
 'cat__job_title_AI Engineer' 'cat__job_title_Backend Developer'
 'cat__job_title_Business Analyst' 'cat__job_title_Cloud Engineer'
 'cat__job_title_Cybersecurity Analyst' 'cat__job_title_Data Analyst'
 'cat__job_title_Data Scientist' 'cat__job_title_DevOps Engineer'
 'cat__job_title_Frontend Developer'
 'cat__job_title_Machine Learning Engineer'
 'cat__job_title_Product Manager' 'cat__job_title_Software Engineer']


REGRESION

In [8]:

modelos = {
    '1. Regresion Lineal': LinearRegression(),
    '2. Ridge': Ridge(alpha=1.0),
    # '3. K-Vecinos (KNN)': KNeighborsRegressor(n_neighbors=5, n_jobs=-1), # tarda una locura no hacer
    '4. Random Forest': RandomForestRegressor(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1),
    '5. Gradient Boosting': GradientBoostingRegressor(n_estimators=80, random_state=42)
}

resultados = []

for nombre, algoritmo in modelos.items():
    print(f'Entrenando {nombre}...')

    # entrenar
    algoritmo.fit(X_train_transformed, y_train)

    # predecir
    y_pred = algoritmo.predict(X_test_transformed)

    # comparar
    mse_test = mean_squared_error(y_test, y_pred)
    r2_test = r2_score(y_test, y_pred)

    # guardar resultados
    resultados.append({
        'Modelo': nombre,
        'Test R2': r2_test,
        'Test MSE': mse_test
    })

    print(f" -> Test R2: {r2_test:.4f} | Test MSE: {mse_test:,.2f}\n")

# Resultados
resultados_df = pd.DataFrame(resultados).sort_values(by='Test R2', ascending=False).reset_index(drop=True)
display(resultados_df)

print(f"Mejor modelo base según Test R2: {resultados_df.loc[0, 'Modelo']}")

Entrenando 1. Regresion Lineal...
 -> Test R2: 0.9635 | Test MSE: 50,773,076.89

Entrenando 2. Ridge...
 -> Test R2: 0.9635 | Test MSE: 50,772,652.46

Entrenando 4. Random Forest...
 -> Test R2: 0.6496 | Test MSE: 487,037,828.75

Entrenando 5. Gradient Boosting...
 -> Test R2: 0.9283 | Test MSE: 99,655,083.84



,Modelo,Test R2,Test MSE
0,2. Ridge,0.963469,5.077265e+07
1,1. Regresion Lineal,0.963469,5.077308e+07
2,5. Gradient Boosting,0.928299,9.965508e+07
3,4. Random Forest,0.649579,4.870378e+08


Mejor modelo base según Test R2: 2. Ridge
